In [1]:
# =========================
# 1. Install dependenicies
# =========================


%pip install -U gradio pypdf sentence-transformers transformers torch scikit-learn


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# =========================
# 2. Imports
# =========================

import gradio as gr
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from sklearn.metrics.pairwise import cosine_similarity


C:\Users\donal\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# =========================
# 3. Load PDF
# =========================

PDF_PATH =  r"C:\Users\donal\OneDrive\Documents\Documents\Stylesense-AI\fashsion.pdf"  

reader = PdfReader(PDF_PATH)

pages = []
for_page = 0

for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text:
        pages.append({
            "page": i + 1,
            "text": text
        })

print(f"Loaded {len(pages)} pages")


Loaded 13 pages


In [4]:
# =========================
# 4. Split text into chunks
# =========================

def split_text(text, chunk_size=900, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap

    return chunks


chunks = []

for page in pages:
    for chunk in split_text(page["text"]):
        chunks.append({
            "page": page["page"],
            "text": chunk
        })

print(f"Created {len(chunks)} chunks")


Created 39 chunks


In [5]:
# =========================
# 5. Create embeddings
# =========================

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_embeddings = embed_model.encode(chunk_texts, show_progress_bar=True)


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.39it/s]


In [6]:
# =========================
# 6. Load CPU chat model
# =========================

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

chat_model_name = "Qwen/Qwen2.5-0.5B-Instruct"

chat_tokenizer = AutoTokenizer.from_pretrained(chat_model_name)
chat_model = AutoModelForCausalLM.from_pretrained(
    chat_model_name,
    torch_dtype=torch.float32,
    device_map=None
)

chat_model.eval()

print("Chat model loaded")



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:03<00:00, 93.86it/s] 


Chat model loaded


In [7]:
# =========================
# 7. Retrieve relevant PDF context
# =========================

def retrieve_context(user_query, top_k=4):
    question_embedding = embed_model.encode([user_query])
    similarities = cosine_similarity(question_embedding, chunk_embeddings)[0]
    best_indices = similarities.argsort()[-top_k:][::-1]

    selected_chunks = [chunks[idx]["text"] for idx in best_indices]
    return "\n\n".join(selected_chunks)


In [8]:
def format_history(history):
    conversation = ""

    if not history:
        return conversation

    for item in history[-6:]:
        if isinstance(item, dict):
            role = item.get("role", "")
            content = item.get("content", "")

            if role == "user":
                conversation += f"User: {content}\n"
            elif role == "assistant":
                conversation += f"Assistant: {content}\n"

        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            user_msg = item[0]
            bot_msg = item[1]
            conversation += f"User: {user_msg}\nAssistant: {bot_msg}\n"

    return conversation


def recommend_from_pdf(user_query, history=None):
    context = retrieve_context(user_query, top_k=4)
    conversation = format_history(history)

    messages = [
        {
            "role": "system",
            "content": (
                "You are StyleSense AI, a friendly fashion recommendation chatbot. "
                "Give practical outfit, dress, color, accessory, and styling suggestions. "
                "Use the fashion reference as background knowledge, but do not mention PDF, source pages, chunks, or documents. "
                "Ask a short follow-up question only if the user's request is too vague. "
                "Do not repeat the user's question. Answer naturally like a stylist."
            )
        },
        {
            "role": "user",
            "content": f"""
Fashion reference:
{context}

Recent conversation:
{conversation}

User message:
{user_query}

Give a helpful fashion recommendation.
"""
        }
    ]

    prompt = chat_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = chat_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1800
    )

    with torch.no_grad():
        output = chat_model.generate(
            **inputs,
            max_new_tokens=220,
            temperature=0.75,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=chat_tokenizer.eos_token_id
        )

    generated_tokens = output[0][inputs["input_ids"].shape[-1]:]
    answer = chat_tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return answer.strip()


In [ ]:
import gradio as gr
import base64

# =========================
# Load Background Image
# =========================
def image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode()

bg_image = image_to_base64(
    r"C:\Users\donal\OneDrive\Documents\Documents\Stylesense-AI\download.jpg"
)

# =========================
# Chat Function
# =========================
def respond(message, history):
    bot_reply = recommend_from_pdf(message, history)

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": bot_reply})

    return "", history

# =========================
# CSS (UI Design)
# =========================
custom_css = f"""
@import url('https://fonts.googleapis.com/css2?family=Great+Vibes&family=Poppins:wght@300;400;600;700;800&display=swap');

.gradio-container {{
    background:
        linear-gradient(rgba(10, 10, 10, 0.25), rgba(10, 10, 10, 0.55)),
        url("data:image/jpeg;base64,{bg_image}") !important;
    background-size: cover !important;
    background-position: center top !important;
    background-repeat: no-repeat !important;
    min-height: 1500px !important;
    max-width: 100% !important;
    padding: 0 !important;
    font-family: 'Poppins', sans-serif !important;
}}

#main {{
    max-width: 980px;
    margin: auto;
    padding-top: 50px;
}}

#brand {{
    text-align: center;
    color: white;
}}

#brand-title {{
    font-family: 'Great Vibes', cursive !important;
    font-size: 90px;
    color: #ff8fbd;
    text-shadow: 0 0 20px #ff4fa3;
}}

#ai {{
    font-size: 45px;
    color: #ffe8bd;
    font-weight: bold;
}}

#tagline {{
    color: #f5d7a6;
    letter-spacing: 4px;
    font-size: 16px;
    font-weight: 700;
}}

#subtitle {{
    color: white;
    font-size: 18px;
    margin-top: 15px;
}}

#try-box {{
    background: rgba(20, 18, 22, 0.9);
    border: 1.5px solid #ff5ca8;
    border-radius: 26px;
    padding: 20px;
    text-align: center;
    margin: 25px auto -30px auto;
    width: 850px;
    z-index: 2;
}}

.prompt {{
    display: inline-block;
    color: white;
    border: 1.5px solid #ff5ca8;
    border-radius: 18px;
    padding: 12px 18px;
    margin: 6px;
    font-weight: 600;
}}

#chat-card {{
    background: rgba(255,255,255,0.97);
    border-radius: 32px;
    padding: 30px;
    margin-top: 30px;
    box-shadow: 0 20px 60px rgba(0,0,0,0.5);
}}

#features {{
    display: grid;
    grid-template-columns: repeat(4,1fr);
    background: rgba(20,18,22,0.9);
    border-radius: 0 0 25px 25px;
    padding: 25px;
}}

.feature {{
    text-align: center;
    color: white;
}}

.feature h3 {{
    color: #ff5ca8;
}}

#footer {{
    text-align: center;
    margin-top: 20px;
    color: white;
}}

textarea {{
    border-radius: 15px !important;
}}

button {{
    background: #ff4fa3 !important;
    color: white !important;
}}
"""

# =========================
# UI Layout
# =========================
with gr.Blocks(css=custom_css) as demo:

    gr.HTML("""
    <div id="main">

        <div id="brand">
            <div id="brand-title">StyleSense</div>
            <div id="ai">AI</div>
            <div id="tagline">YOUR PERSONAL AI FASHION STYLIST</div>
            <div id="subtitle">
                Get outfit ideas, styling tips, color combinations and accessories.
            </div>
        </div>

        <div id="try-box">
            <div class="prompt">👗 Wedding outfit</div>
            <div class="prompt">💼 Office outfit</div>
            <div class="prompt">💎 Black dress styling</div>
        </div>

        <div id="chat-card">
    """)

    chatbot = gr.Chatbot(
        height=400,
        value=[
            {
                "role": "assistant",
                "content": "✨ Hi there! 👋 I'm StyleSense AI.\nAsk me anything about outfits!"
            }
        ]
    )

    with gr.Row():
        msg = gr.Textbox(placeholder="Ask your styling question...", scale=8)
        btn = gr.Button("➤", scale=1)

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    btn.click(respond, [msg, chatbot], [msg, chatbot])

    gr.HTML("""
        </div>

        <div id="features">
            <div class="feature"><h3>👗 Outfit Ideas</h3></div>
            <div class="feature"><h3>🎨 Color Match</h3></div>
            <div class="feature"><h3>👜 Accessories</h3></div>
            <div class="feature"><h3>✨ Fashion Tips</h3></div>
        </div>

        <div id="footer">
            Be You. Be Stylish. Be Confident.
        </div>

    </div>
    """)

demo.launch(debug=True)

C:\Users\donal\AppData\Local\Temp\ipykernel_38800\2193782381.py:147: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
